# Postprocessing

In this notebook we take a deeper look at postprocessing, using the python interface of GVEC.

GVEC is parallelised with OpenMP and we can set the number of OpenMP threads before importing the `gvec` package.

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "2"

This tutorial requires `matplotlib` to be installed in addition to `gvec`. `numpy` and `xarray` are dependencies of `gvec` and should therefore be already installed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import gvec

In this tutorial we will use the equilibrium of the elliptic stellarator, computed in the tutorial [](./020_stellarator)([&#x1F310;](https://gvec.readthedocs.io/latest/tutorials/notebooks/020_stellarator.html)).

To obtain a [`gvec.State`](#gvec.core.state.State)([&#x1F310;](https://gvec.readthedocs.io/latest/api/core-state.html#gvec.core.state.State)) object for this equilibrium, [`gvec.load_state`](#gvec.core.state.load_state)([&#x1F310;](https://gvec.readthedocs.io/latest/api/core-state.html#gvec.core.state.load_state)) can be used with specific paths for the [parameter](<path:./run_ellipstell/parameter_GVEC_ellipstell_final.ini>) and [statefile](<path:./run_ellipstell/GVEC_ellipstell_State_final.dat>).
Alternatively, when each equilibrium is saved in a separate directory, [`gvec.find_state`](#gvec.core.state.find_state)([&#x1F310;](https://gvec.readthedocs.io/latest/api/core-state.html#gvec.core.state.find_state])) can load the state with only the directory provided.

In [ ]:
state = gvec.find_state("./run_ellipstell/")

## Derived Quantities
One of the central features of the GVEC python bindings is the ability to compute a variety of derived quantities.
A list of computable quantities is returned by `gvec.table_of_quantities(markdown=True)` and also shown [at the end of this notebook](#available-quantities-for-evaluation).

The requested quantities will be stored in an [`xarray.Dataset`](<inv:xarray#xarray.Dataset>), which groups several varaibles, their coordinates and metadata, similar to a *pandas Dataframe*.
Such a `Dataset` can also be stored directly as *netCDF*.
GVEC will also automatically determine which quantities are necessary to compute the desired quantities and add them to the `Dataset`, thereby also caching them for future computations.

The grid on which the quantities are to be evaluated, can be specified explicitly (with an array or float) or automatically with either an integer for linear spacing (within one field period) or `"int"` for the integration points required by GVEC.

The [`state.evaluate`](#gvec.core.state.State.evaluate)([&#x1F310;](https://gvec.readthedocs.io/latest/api/core-state.html#gvec.core.state.State.evaluate)) method can be used to create a new `Dataset` and compute desired quantities in a single step:

In [ ]:
ev = state.evaluate("pos", "B", rho=[0.1, 0.5, 0.9], theta=20, zeta=0.0)
ev

An existing dataset can be extended using [`state.compute`](#gvec.core.state.State.compute)([&#x1F310;](https://gvec.readthedocs.io/latest/api/core-state.html#gvec.core.state.State.compute)):

In [ ]:
state.compute(ev, "J", "V")

The individual quantities, within an [`xarray.Dataset`](<inv:xarray#xarray.Dataset>) can be accessed using `ev.B` or `ev["B"]` and are enriched with coordinate information which can be used to a variety of operations, e.g.:

In [ ]:
mean_B = ev.B.mean(dim=("rad", "pol", "tor"))
B2 = xr.dot(ev.B, ev.B, dim="xyz")
JxB = xr.cross(ev.J, ev.B, dim="xyz")

Indexing is best done using `ev.B.sel(rho=0.5)` (by value) or `ev.B.isel(rad=0)` (by position)
and the raw `numpy` array can be extracted with `ev.B.values`.
To ensure a particular order, use `ev.B.transpose("rad", "pol", "tor", "xyz").values`.
To convert a scalar (e.g. the volume `V`) into a python scalar use the `ev.V.item()` method.
The `.squeeze()` method can be used to remove any dimensions of length 1.

For more information, see the [xarray documentation](https://docs.xarray.dev/en/stable/).

## Computing integrals

Specifying `"int"` for `rho`, `theta` and `zeta` in the `Evaluations` function, chooses the grid to be the integration points used by GVEC internally.
The functions `radial_integral`, `fluxsurface_integral` and `volume_integral` can then be used to perform integration with the apropriate weights.
E.g. to compute the volume averaged plasma beta, one could use:

In [ ]:
ev = state.evaluate("mod_B", "mu0", "p", "Jac", "V", rho="int", theta="int", zeta="int")
beta = ev.p / (ev.mod_B**2 / (2 * ev.mu0))
beta_avg = gvec.volume_integral(beta * ev.Jac) / ev.V
beta_avg.item()

For supported quantities which require integration (e.g. the volume `V` computed above), an auxiliary dataset is created to compute them if the grid does not conform to the integration points.

## Boozer transform

To evaluate the equilibrium in Boozer angles, you can use [`state.evaluate_sfl(..., sfl="boozer")`](#gvec.core.state.State.evaluate_sfl)([&#x1F310;](https://gvec.readthedocs.io/latest/api/core-state.html#gvec.core.state.State.evaluate_sfl)), which performs a Boozer transform to obtain a set of $\vartheta,\zeta$ points which correspond to your desired grid in $\vartheta_B,\zeta_B$.
The evaluations with this new dataset work the same as above, note however that the suffixes `t` and `z` still refer to components/derivatives with respect to $\vartheta,\zeta$.
Some additional quantities, like `B_contra_t_B` or `e_zeta_B` are now also available.

The argument `MNfactor` specifies how many fourier modes should be used for the Boozer potential $\nu_B$ (and recomputed straight fieldline potential $\lambda$) relative to maximum fourier modes used for computing the equilibrium solution.
This parameter has a strong influence on the accuracy and required computational effort!

In [ ]:
ev = state.evaluate_sfl(
    "mod_B",
    "B_contra_t_B",
    "B_contra_z_B",
    rho=1.0,
    theta=40,
    zeta=50,
    sfl="boozer",
).squeeze()

fig, axs = plt.subplots(1, 2, figsize=(15, 6), layout="constrained")
fig.suptitle(r"Streamlines of $\mathbf{B}$ in boozer and logical coordinates at $\rho=1.0$.")

streamplot_kwargs = dict(
    color="black",
    broken_streamlines=False,
    density=1,
    integration_direction="both",
    start_points=np.vstack(
        [
            np.linspace(0, 2 * np.pi / state.nfp, 22)[1:-1],
            np.linspace(2 * np.pi, 0, 22)[1:-1],
        ]
    ).T,
)

ax = axs[0]
c = ax.contour(ev.zeta_B, ev.theta_B, ev.mod_B, levels=21, cmap="plasma")
fig.colorbar(c, ax=[axs[0], axs[1]], label=f"${ev.mod_B.attrs['symbol']}$")
ax.streamplot(
    ev.zeta_B.values,
    ev.theta_B.values,
    ev.B_contra_z_B,
    ev.B_contra_t_B,
    **streamplot_kwargs,
)
ax.set_xlabel(r"$\zeta_B$")
ax.set_ylabel(r"$\theta_B$")
ax.set_title("Grid in boozer coordinates")

ev = state.evaluate("mod_B", "B_contra_t", "B_contra_z", rho=1.0, theta=40, zeta=50).squeeze()

ax = axs[1]
ax.contour(ev.zeta, ev.theta, ev.mod_B, levels=21, cmap=c.cmap, norm=c.norm)
ax.streamplot(
    ev.zeta.values,
    ev.theta.values,
    ev.B_contra_z.values,
    ev.B_contra_t.values,
    **streamplot_kwargs,
)
ax.set_xlabel(r"$\zeta$")
ax.set_ylabel(r"$\theta$")
ax.set_title("Grid in logical coordinates");

:::{note}

The Boozer transform recomputes $\lambda$ with a higher resolution (to satisfy the integrability condition for $\nu_B$)!
Therefore some quantities will differ between the equilibrium evaluation and Boozer evaluation.

In particular $\langle B_\vartheta \rangle, \langle B_\zeta \rangle$ will differ from $B_{\vartheta_B},B_{\zeta_B}$ by an offset.

:::
:::{note}

Currently the Boozer transform is performed for each surface individually and radial derivatives are therfore not available.
This means that that $\frac{\partial \mathbf{B}}{\partial \rho}$ and $\mathbf{J}$ are not available!

:::

## Field-aligned grid
The `state.evaluate_sfl` method and `EvaluationsBoozer` factory function can also be used to generate a non-tensorproduct grid by providing (up to 3D) arrays for the values of $\theta_B,\zeta_B$.
This can be used for example to create a field-aligned grid:

In [ ]:
rho = [0.5, 1.0]  # radial positions
alpha = np.linspace(0, 2 * np.pi, 100, endpoint=False)  # fieldline label
phi = np.linspace(0, 2 * np.pi / state.nfp, 101)  # angle along the fieldline

# evaluate the rotational transform (fieldline angle) on the desired surfaces
iota = state.evaluate("iota", rho=rho, theta=None, zeta=None).iota

# 3D toroidal and poloidal arrays that correspond to fieldline coordinates for each surface
theta_B = alpha[None, :, None] + iota.data[:, None, None] * phi[None, None, :]

# create the grid
ev = gvec.EvaluationsBoozer(rho=rho, theta_B=theta_B, zeta_B=phi, state=state, MNfactor=5)

# set the fiedline label as poloidal coordinate & index (not necessary, but good practice)
ev["alpha"] = ("pol", alpha)
ev["alpha"].attrs = dict(symbol=r"\alpha", long_name="fieldline label")
ev = ev.set_coords("alpha").set_xindex("alpha")

state.compute(ev, "B", "B_contra_t_B", "B_contra_z_B", "mod_B")

In [ ]:
ev = ev.sel(rho=0.5)

fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")

c = ax.contour(ev.zeta_B, ev.alpha, ev.mod_B, levels=21, cmap="plasma")
fig.colorbar(c, ax=ax, label=f"${ev.mod_B.attrs['symbol']}$")
ax.set_xlabel(f"${ev.zeta_B.attrs['symbol']}$")
ax.set_ylabel(f"${ev.alpha.attrs['symbol']}$")
ax.set_title(f"${ev.mod_B.attrs['symbol']}$ on a field-aligned grid at $\\rho=0.5$");

---
## Available quantities for evaluation
The following table contains the quantities that can be evaluated with the python bindings, it can be generated with 
```python
gvec.table_of_quantities(markdown=True)
```
```{include} ../../generators/quantities.md
